# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields, per Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields using the `@id` attributes.

In [ ]:
# List available Record Sets and their Field @ids
if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        print(f"RecordSet @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '')}")
        print("  Fields:")
        for field in record_set['fields']:
            print(f"    - {field['@id']} (name: {field['name']}, type: {field.get('dataType', '')})")
        print('')
else:
    # Fallback: introspect dataset (for compatibility)
    print("Record sets not found as 'record_sets' attribute. Trying to infer from dataset.records...")
    # Try to enumerate record set name guesses
    try:
        for record_set_id in dataset.record_set_ids:
            print(f"Found RecordSet: {record_set_id}")
            records = list(dataset.records(record_set=record_set_id))
            if len(records) > 0:
                print(f"Sample keys: {list(records[0].keys())}")
    except Exception as e:
        print(f"Error enumerating record sets: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s you just observed.

In [ ]:
# List available record set @ids
record_set_ids = dataset.record_set_ids
print(f"Record set @ids in this dataset: {record_set_ids}")

# Choose a record set for extraction (replace with your chosen @id)
chosen_record_set = record_set_ids[0]
print(f"Chosen record set: {chosen_record_set}")

# Load all records as DataFrame
records = list(dataset.records(record_set=chosen_record_set))
df = pd.DataFrame(records)
print(f"Columns in {chosen_record_set}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing on a chosen numeric field. All fields are referenced by their `@id`, as per mlcroissant conventions.

In [ ]:
# List all fields (columns) and choose a numeric one for analysis
print(df.dtypes)
# 
# Example: Use first float or int column as numeric field
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    # Try to convert columns with digits to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_fields.append(col)
        except:
            pass
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using '{numeric_field}' as the numeric field (by @id)")

    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Try grouping by another field
    group_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < 10 and df[c].dtype == object]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field you analyzed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion
Using `mlcroissant`, we've loaded the FAIR² mass spectrometry EV dataset, explored its structure via `@id` references, extracted tabular data, and performed basic statistical and visualization operations. For deeper analysis, map dataset metadata and fields directly from the schema using their `@id` for best reproducibility.